# Notebook 2 — Baseline vs Enhanced Pipeline (Demo Log)

**Objective**: side-by-side comparison of the naive baseline RAG and the enhanced corrective RAG (CRAG) pipeline on six representative queries that together exercise every behaviour the enhanced pipeline implements.

This notebook is the source for `demo_log.pdf` (spec 09 deliverable).

**Format**: one **smoke test** cell that live-runs the enhanced pipeline (proves the code works end-to-end), then six **showcase** queries that load locked answers from `data/evaluation_results/` so the answers shown here are exactly the answers RAGAS scored in NB3. This guarantees the demo log and the report's metric tables refer to the same answers.

**Behaviours showcased**:

| # | qid | Behaviour |
|---|-----|-----------|
| 1 | q11 | Section-aware chunking + box-analysis metadata filter (enhanced wins) |
| 2 | q01 | Both pipelines succeed; rerank changes top-1 chunk ordering |
| 3 | q15 | CRAG corrective loop: rewrite fires on retrieval failure and recovers |
| 4 | q24 | Honest limitation: both pipelines struggle with a page-number citation |
| 5 | q21 | Out-of-corpus scope gate: enhanced abstains rather than hallucinate |
| 6 | q13 | Hallucination check fires + rerank changes top-1 |

## Setup

In [1]:
import json, os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "data" / "evaluation_results"

baseline_results = json.loads((RESULTS / "baseline_results.json").read_text())
enhanced_results = json.loads((RESULTS / "enhanced_results.json").read_text())

print(f"Loaded results for {len(baseline_results)} baseline queries and "
      f"{len(enhanced_results)} enhanced queries.")

def _trim(s: str, n: int = 350) -> str:
    s = (s or "").strip().replace("\n", " ")
    return s if len(s) <= n else s[:n].rstrip() + " […]"

def show_query(qid: str, behaviour: str):
    b = baseline_results[qid]
    e = enhanced_results[qid]
    print("=" * 88)
    print(f"  {qid}  |  {behaviour}")
    print("=" * 88)
    print(f"\nQuestion:  {b['question']}")
    print(f"Category:  {b['category']}\n")

    print("--- BASELINE -------------------------------------------------")
    print(f"  trace:           {b['pipeline_trace']}")
    print(f"  chunks used:     {b['chunks_used']}")
    print(f"  is_grounded:     {b['is_grounded']}")
    print(f"  answer:\n      {_trim(b['answer'], 600)}\n")

    print("--- ENHANCED -------------------------------------------------")
    print(f"  trace:           {e['pipeline_trace']}")
    print(f"  chunks used:     {e['chunks_used']}")
    print(f"  rewrites:        {e['crag_rewrites']}")
    print(f"  halluc retries:  {e['hallucination_retries']}")
    print(f"  is_grounded:     {e['is_grounded']}")
    print(f"  filters used:    {e['metadata_filters_used']}")
    pre, post = e.get('pre_rerank_ids') or [], e.get('post_rerank_ids') or []
    if pre and post:
        flipped = "yes" if pre[0] != post[0] else "no"
        print(f"  rerank flipped top-1: {flipped}")
        print(f"    pre-rerank ids:  {pre[:5]}")
        print(f"    post-rerank ids: {post[:5]}")
    print(f"  answer:\n      {_trim(e['answer'], 600)}")


Loaded results for 25 baseline queries and 25 enhanced queries.


## Smoke test — live execution of the enhanced pipeline

One live call to prove the code path is intact end-to-end. Uses the simplest factual query in the test set (q02). All other showcase cells below load locked outputs.

In [2]:
# q02: simple factual, ~3 API calls (Anthropic + OpenAI + Cohere), <$0.05.
# Wrapped in try/except so transient API issues don't break the notebook.
try:
    from boe_rag.pipelines.enhanced import EnhancedPipeline
    pipe = EnhancedPipeline()
    smoke_question = enhanced_results["q02"]["question"]
    print(f"Question: {smoke_question}\n")
    result = pipe.run(smoke_question)
    print(f"pipeline_trace: {result.pipeline_trace}")
    print(f"chunks_used:    {result.chunks_used}")
    print(f"is_grounded:    {result.is_grounded}\n")
    print(f"Answer:\n  {result.answer[:500]}")
except Exception as exc:
    print(f"[smoke test skipped] live API call failed: {type(exc).__name__}: {exc}")
    print("All showcase cells below use locked results from data/evaluation_results/")


Question: What was Brent crude price cited in the March 2026 MPC minutes?



pipeline_trace: ('analyze_query', 'retrieve', 'grade_documents', 'rerank', 'generate', 'check_hallucination')
chunks_used:    1
is_grounded:    True

Answer:
  According to the provided document, paragraph 3 states: "The Brent crude spot price in the run-up to the MPC meeting on 18 March had been over $100 per barrel."

The document indicates this was around 60% higher than at the time of the February Report and represented the highest level since 2022.


## Showcase 1 — Box D consumption weakness (q11)

**What to notice**: the enhanced pipeline filters retrieval to `section_category=box_analysis`, pulling Box D as a coherent unit. The baseline scatters across unrelated MPR paragraphs and produces a longer but less precise answer. The enhanced pipeline's hallucination check also fires here (trace shows `generate → check_hallucination → generate`) — the retry tightened the answer.

In [3]:
show_query("q11", "Section-aware chunking + box-analysis metadata filter")

  q11  |  Section-aware chunking + box-analysis metadata filter

Question:  What specific consumption weakness scenario did Box D in the November 2025 MPR describe?
Category:  deep_context

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      Based on the context provided, Box D in the November 2025 MPR described a consumption weakness scenario with the following specific elements:  **Key Components of the Lower Demand Scenario:**  1. **Persistent shift in household saving behavior** - Households continuing to save at elevated levels rather than spending, with the saving rate remaining higher than the baseline projection expects  2. **Precautionary savings increase** - Households maintaining higher savings due to:    - Lasting effects from the cost-of-living crisis experience    - Increased concerns about employment prospects    - […]

--- ENHANCED ------------------------

## Showcase 2 — MPC vote split February 2026 (q01)

**What to notice**: both pipelines succeed (vote splits are simple factual content that lives in many chunks). The interesting bit for the enhanced pipeline is the rerank: pre-rerank top-1 is one chunk, post-rerank top-1 is a different one. Cohere's relevance model surfaces a more directly-on-question chunk than vector cosine similarity alone.

In [4]:
show_query("q01", "Both pipelines succeed; rerank flips top-1")

  q01  |  Both pipelines succeed; rerank flips top-1

Question:  What was the MPC vote split in February 2026?
Category:  simple_factual

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      Based on the context provided, in February 2026 the MPC vote split was **5-4 to maintain Bank Rate at 3.75%**.  Specifically: - **5 members voted to maintain Bank Rate at 3.75%**: Andrew Bailey, Megan Greene, Clare Lombardelli, Catherine L Mann, and Huw Pill - **4 members voted to reduce Bank Rate by 0.25 percentage points to 3.5%**: Sarah Breeden, Swati Dhingra, Dave Ramsden, and Alan Taylor  This information is clearly stated in both the Monetary Policy Summary and the detailed voting records in paragraphs 16, 17, and 19 of the minutes.

--- ENHANCED -------------------------------------------------
  trace:           ['analyze_query', 'retrieve', 'grade_documents', 'rerank', 'genera

## Showcase 3 — Lombardelli asymmetric policy risk (q15)

**What to notice**: the enhanced pipeline's first retrieval pass found nothing the grader judged relevant. The CRAG corrective loop fired: `analyze_query → retrieve → grade_documents → rewrite_query → retrieve → grade_documents → rerank → generate`. The rewritten query recovered relevant chunks and produced a grounded answer.

In [5]:
show_query("q15", "CRAG corrective loop: rewrite fires and recovers")

  q15  |  CRAG corrective loop: rewrite fires and recovers

Question:  What asymmetric policy risk argument did Lombardelli make in November 2025?
Category:  deep_context

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      Based on the context provided, Lombardelli made an asymmetric policy risk argument in November 2025 centered around the risk of being "too late" rather than "too early" with monetary policy easing.  Specifically, Lombardelli argued that:  1. **The risk of undershooting is greater than overshooting**: If policymakers misjudge the economic situation or are too slow to recognize that a demand-stalling scenario (case 1) is developing, they would end up "using case 2 policies in a case 1 world."  2. **Consequences of being too restrictive are severe**: This policy mistake would risk:    - Undershoo […]

--- ENHANCED -----------------------------------------

## Showcase 4 — GDP Q3 2025, page 23 (q24) — honest limitation

**What to notice**: the question asks for an exact figure from a specific page of a specific document. Both pipelines struggle because (a) embeddings don't preserve numerical precision and (b) chunks don't preserve page numbers. The enhanced pipeline ultimately abstains (`abstain` in the trace) rather than fabricate a number — which is the right behaviour, even though the assignment ground truth says this question is in-corpus. This is one of three false-positive abstains contributing to the 0.25 abstain-precision noted in NB3.

In [6]:
show_query("q24", "Both pipelines struggle; enhanced abstains rather than fabricate")

  q24  |  Both pipelines struggle; enhanced abstains rather than fabricate

Question:  What was the exact GDP growth figure for Q3 2025 cited on page 23 of the November MPR?
Category:  edge_case_numerical

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      I cannot find the specific GDP growth figure for Q3 2025 that was cited on page 23 of the November MPR in the provided context.   The context does mention that "GDP growth had eased to 0.1% in 2025 Q3, slightly below the rate expected in the November Report," but it doesn't specify what the exact figure cited in the November Report was - only that the actual 0.1% growth was "slightly below" that expectation.  To answer your question about the exact figure from page 23 of the November MPR, I would need access to that specific document and page.

--- ENHANCED -------------------------------------------------
  trace:    

## Showcase 5 — Federal Reserve view on rates (q21) — out-of-corpus

**What to notice**: the question is about the Fed, not the Bank of England. The B1 scope-detection extension flags `out_of_corpus=True` in `analyze_query`, and the router short-circuits straight to `abstain_out_of_corpus` — no retrieval, no generation, no API spend on a question the corpus cannot answer. Trace: `analyze_query → abstain_out_of_corpus`. This is the should-abstain-recall=1.00 behaviour.

In [7]:
show_query("q21", "Out-of-corpus scope gate: enhanced abstains")

  q21  |  Out-of-corpus scope gate: enhanced abstains

Question:  What is the Federal Reserve's view on interest rates?
Category:  edge_case_out_of_scope

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      Based on the context provided, the Federal Reserve's view on interest rates can be inferred from their recent actions and market expectations:  **Recent Actions:** - The Federal Reserve maintained the target range for the federal funds rate at 4.25%-4.5% at its March meeting - They announced a slower pace for their quantitative tightening programme  **Market Expectations of Fed Policy:** - Markets initially expected the Federal Reserve would keep the federal funds rate higher in the near term to counteract expected inflationary pressures from:   - Fiscal stimulus   - Trade restrictions    - Do […]

--- ENHANCED -------------------------------------------------
  trace:

## Showcase 6 — US corporate default risks (q13)

**What to notice**: this query exercises two safety mechanisms in sequence. Cohere reranking changes the top-1 chunk, and the hallucination check fires after the first generation (trace shows `generate → check_hallucination → generate → check_hallucination`). The retry produced a grounded final answer.

In [8]:
show_query("q13", "Hallucination check fires + rerank flips top-1")

  q13  |  Hallucination check fires + rerank flips top-1

Question:  What risks from US corporate defaults did the December 2025 FSR identify?
Category:  deep_context

--- BASELINE -------------------------------------------------
  trace:           ['retrieve', 'generate']
  chunks used:     5
  is_grounded:     None
  answer:
      Based on the context provided, the December 2025 FSR identified several key risks from US corporate defaults:  ## Specific Default Cases and Their Implications  The FSR highlighted two high-profile US corporate defaults that occurred in September: - **First Brands Group** (automotive parts supplier) - **Tricolor** (subprime auto loan provider)  ## Key Risks Identified  ### 1. **Systemic Market Risks** - Potential for **spillovers to interconnected markets** and other forms of risky credit - Risk of reduced investor confidence and **widening credit spreads** - Possibility of **tightening financ […]

--- ENHANCED ---------------------------------------------

## Cross-query summary

In [9]:
rows = []
for qid in ["q11", "q01", "q15", "q24", "q21", "q13"]:
    e = enhanced_results[qid]
    pre, post = e.get('pre_rerank_ids') or [], e.get('post_rerank_ids') or []
    rerank_flipped = (bool(pre) and bool(post) and pre[0] != post[0])
    rows.append({
        "qid": qid,
        "category": e["category"],
        "trace_steps": len(e["pipeline_trace"]),
        "rewrite_fired": "rewrite_query" in e["pipeline_trace"],
        "rerank_flipped": rerank_flipped,
        "halluc_retry": e["hallucination_retries"] > 0,
        "abstained": "abstain" in e["pipeline_trace"] or "abstain_out_of_corpus" in e["pipeline_trace"],
        "is_grounded": e["is_grounded"],
        "chunks_used": e["chunks_used"],
    })
display(pd.DataFrame(rows).set_index("qid"))


,category,trace_steps,rewrite_fired,rerank_flipped,halluc_retry,abstained,is_grounded,chunks_used
qid,,,,,,,,
q11,deep_context,8,False,False,True,False,True,1
q01,simple_factual,6,False,True,False,False,True,4
q15,deep_context,9,True,True,False,False,True,3
q24,edge_case_numerical,7,True,False,False,True,None,0
q21,edge_case_out_of_scope,2,False,False,False,True,None,0
q13,deep_context,8,False,True,True,False,True,3


## What's next

Notebook 3 quantifies these behaviours across the full 25-query test set with RAGAS metrics, paired Wilcoxon + Holm-Bonferroni statistical tests, and per-category analysis.